# Can we predict 12-month loan default?
**Question:** using only information available at application time, can we predict `defaulted` (1 = defaulted within 12 months of funding)?

**Plan:** data checks, EDA, baseline (majority class), logistic regression and gradient boosting compared with stratified CV on a training split, then a one-time evaluation on a held-out test set, then interpretation.
Only ~10% of loans default, so accuracy is misleading; we focus on ROC AUC, PR AUC (average precision), calibration and lift.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv'); dd=pd.read_csv('columns.csv')
dd

## 1. Data checks

In [ ]:
print(df.shape)
print('missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
df.describe().T

In [ ]:
y=df.defaulted
print(y.value_counts().to_frame('n').assign(share=lambda t:t.n/len(df)))
print('employment_years > age-16:', (df.employment_years>df.age-16).sum())
print('non-positive income/loan:', (df.income_k<=0).sum(), (df.loan_amount_k<=0).sum())

`application_id` is an identifier and is excluded from the features. There is no application date, so a time-based split is impossible; we use stratified random splits (caveat: performance on future vintages may differ).

In [ ]:
# does the ID carry signal (ordering/leakage)? Expect ~0.5
print('AUC of application_id alone:', round(roc_auc_score(y, df.application_id),3))

## 2. EDA: default rate vs. each feature

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(14,7))
for a,col in zip(ax.ravel(),num):
    q=pd.qcut(df[col],10,duplicates='drop')
    r=df.groupby(q,observed=True).defaulted.mean()
    a.plot(range(len(r)),r.values,marker='o'); a.axhline(y.mean(),ls='--',c='grey')
    a.set_title(col); a.set_xlabel('decile (low to high)'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','size']))

In [ ]:
df[num+['defaulted']].corr(method='spearman')['defaulted'].drop('defaulted').sort_values()

## 3. Models
Hold out 20% as a test set (used once, at the end). Compare models by 5-fold stratified CV on the other 80%.

In [ ]:
feats=num+['region']
X=df[feats]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),['region'])])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic regression':make_pipeline(pre,LogisticRegression(max_iter=1000)),
 'gradient boosting':make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),['region'])]),
        HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=SEED)),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows={}
for n,m in models.items():
    s=cross_validate(m,Xtr,ytr,cv=cv,scoring=['roc_auc','average_precision','neg_brier_score'])
    rows[n]={'ROC AUC':s['test_roc_auc'].mean(),'AUC sd':s['test_roc_auc'].std(),'PR AUC':s['test_average_precision'].mean(),'Brier':-s['test_neg_brier_score'].mean()}
pd.DataFrame(rows).T.round(3)

No-skill reference: ROC AUC 0.5, PR AUC equal to the default rate (~0.10).

## 4. Held-out test evaluation

In [ ]:
res={}; probs={}
for n,m in models.items():
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; probs[n]=p
    res[n]={'ROC AUC':roc_auc_score(yte,p),'PR AUC':average_precision_score(yte,p),'Brier':brier_score_loss(yte,p)}
print(pd.DataFrame(res).T.round(3))
rng=np.random.default_rng(SEED); yt=yte.values
for n in ['logistic regression','gradient boosting']:
    p=probs[n]; b=[]
    for _ in range(1000):
        i=rng.integers(0,len(yt),len(yt)); b.append(roc_auc_score(yt[i],p[i]))
    print(n,'test AUC 95% bootstrap CI:',np.percentile(b,[2.5,97.5]).round(3))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n,p in probs.items():
    if n.startswith('baseline'): continue
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
    fp,mp=calibration_curve(yte,p,n_bins=8,strategy='quantile'); ax[2].plot(mp,fp,marker='o',label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR')
ax[1].axhline(yte.mean(),c='k',ls='--'); ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision')
ax[2].plot([0,.5],[0,.5],'k--'); ax[2].set(title='Calibration',xlabel='predicted',ylabel='observed')
ax[0].legend(); plt.tight_layout(); plt.show()

### Practical value: lift by risk decile (logistic regression, test set)

In [ ]:
t=pd.DataFrame({'p':probs['logistic regression'],'y':yte.values})
t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=range(1,11))
g=t.groupby('decile',observed=True).agg(pred=('p','mean'),actual=('y','mean'),n=('y','size'))
g['lift']=g.actual/t.y.mean(); g.round(3)

In [ ]:
top=t.sort_values('p',ascending=False).head(int(.2*len(t)))
print(f"Top 20% riskiest applications capture {top.y.sum()/t.y.sum():.0%} of all defaults (random would be 20%).")

## 5. What drives the prediction?

In [ ]:
lr=models['logistic regression']
print(pd.Series(lr[-1].coef_[0],index=lr[0].get_feature_names_out()).sort_values().round(3).to_string())
pi=permutation_importance(lr,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
pd.Series(pi.importances_mean,index=feats).sort_values(ascending=False).round(4).to_frame('AUC drop when shuffled')

Numeric coefficients are on standardized features, so their sizes are comparable; region terms are one-hot.

## 6. Conclusion
Figures below come from one run with `SEED=0`; re-run the notebook to reproduce them.

- **Yes, default is moderately predictable.** Logistic regression reaches test ROC AUC ~0.78 (95% bootstrap CI ~0.75-0.81) against 0.50 for the baseline, and PR AUC ~0.31 against a 0.10 base rate. CV results agree (~0.775 +/- 0.014).
- **It is not highly predictable.** At this level of discrimination, many defaulters still look like ordinary applicants. Treat the model as a risk-ranking tool, not an approve/reject oracle.
- **Practical value:** the riskiest 20% of applications hold about half (53%) of all defaults.
- **Gradient boosting does not beat logistic regression** (0.76 vs 0.78 test AUC, within noise), which suggests the signal is mostly additive and roughly linear. Prefer the simpler, more explainable model.
- **Drivers:** `debt_ratio` (higher = riskier) is by far the strongest, then `income_k` (lower = riskier), then `employment_years` (shorter = riskier) and `loan_amount_k` (larger = riskier). `region`, `age` and `credit_lines` add essentially nothing.
- **Caveats:** (1) no application date, so no out-of-time validation; (2) 643 rows have `employment_years` > `age - 16`, which looks implausible and was not corrected (check with the data owner); (3) the decision threshold depends on the relative cost of a missed default vs. a rejected good loan, which was not provided.